# 🏨 Hotel Reservation System 6: Parallel Agents Over Two Hotels

The chain now has **two hotels**, each with its own database. When a guest asks for a room, the coordinator hands
the request to a **mapper** that fans it out to two reservation assistants, one per hotel, running **in
parallel**. A **merge node** joins their answers into one reply. Getting that fan-out/fan-in right takes three
attempts, and each failure teaches a LangGraph rule.

## Learning Objectives
1. **Parallel writes need reducers** - Two nodes writing one plain key in the same step is an error
2. **One message key per branch** - Why two tool loops cannot share `messages`, and how `ToolNode(messages_key=...)`
   and `partial(tools_condition, messages_key=...)` separate them
3. **A real join** - Sync nodes plus `add_edge([a_done, b_done], "merge_answers")` make the merge node wait for both branches
4. **Proving it ran** - Reading each branch's key, the node trace and both databases to check what happened

## Where this fits
`05_hotel_testing_the_system.ipynb` tested the single-hotel system from `03_hotel_compliance_checker.ipynb`.
This notebook takes that core from `hotel_common` (coordinator + compliance checker) and replaces the one
reservation agent with two parallel ones. `07_hotel_merging_and_conversation_memory.ipynb` lets an LLM merge their answers and keeps the
growing conversation in check. (The date checks of 04 and the confidence score of 05 are left out to keep the
focus on orchestration.)

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import operator
from functools import partial
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, RemoveMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import REMOVE_ALL_MESSAGES, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🏨 Part 1: Two Hotels, Two Databases

`hc.init_two_hotels()` builds one database per hotel from `hc.HOTEL_ROOMS` and the sample bookings for those rooms:
- **Hotel 1** (`hotel1.db`): rooms 101–104 and Alice's booking
- **Hotel 2** (`hotel2.db`): rooms 105–108 and the bookings of Bob (Family Suite, today+30 to today+33) and Charlie

Each hotel gets its own set of SQL tools from `hc.make_sql_tools()`. The tool **names** are identical in both sets
(`run_query`, ...), only the database behind them differs, which is exactly why mixing them up goes unnoticed. The
last lines check in code that each set really reads its own database, by asking each `run_query` for its lowest
and highest room number. The sets are **read-only** (`can_book=False`, so no `book_room`): both assistants run at
the same time, and if both could book, one request would create two reservations.

`hc.free_rooms()` computes the right answer straight from SQLite, so we can compare the agents' answers against
it later. The compliance retriever from notebook 03 is built here too.

In [ ]:
# ============================================================================
# TWO HOTEL DATABASES AND THEIR READ-ONLY TOOLS
# ============================================================================
HOTELS, HOTEL_ROOMS = hc.HOTELS, hc.HOTEL_ROOMS  # {"hotel1": "Hotel 1", ...}; rooms 101-104 and 105-108
db_paths = hc.init_two_hotels()                  # {"hotel1": path, "hotel2": path}
for hotel, rooms in HOTEL_ROOMS.items():
    print(f"✅ {HOTELS[hotel]} ({db_paths[hotel]})")
    for number, room_type, price, capacity, _ in rooms:
        print(f"   room {number}: {room_type:<16} ${price:>5.0f}  sleeps {capacity}")
    for guest, number, start, end in hc.sample_reservations():
        if number in {room[0] for room in rooms}:
            print(f"   booked: room {number} from {start} to {end} ({guest})")

# can_book=False: the parallel assistants report availability; they never book
sql_tools = {hotel: hc.make_sql_tools(db_paths[hotel], can_book=False) for hotel in HOTELS}
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]

print(f"\nTool names, the same in both sets: {[t.name for t in sql_tools['hotel1']]}")
for hotel, rooms in HOTEL_ROOMS.items():
    run_query = next(t for t in sql_tools[hotel] if t.name == "run_query")
    answer = run_query.invoke({"sql": "SELECT MIN(room_number) AS first, MAX(room_number) AS last FROM rooms"})
    expected = f"{rooms[0][0]} | {rooms[-1][0]}"
    print(f"{'✅' if answer.splitlines()[-1] == expected else '❌'} {hotel}'s run_query reads {db_paths[hotel]}: "
          f"rooms {answer.splitlines()[-1]} (expected {expected})")

---
## 📏 Part 2: Rule 1 - Parallel Writes Need a Reducer

Before any LLM is involved, some tiny stand-in graphs (no model calls, same result every run) show the rules.
Here `START` fans out to two nodes that run in the **same step** and both write one key:
- with a plain key (`last_writer: str`) LangGraph cannot choose between the two values and raises `InvalidUpdateError`
- with a reducer (`Annotated[list[str], operator.add]`) both values are kept

A per-branch score such as `confidence: int` hits exactly this error; a list with `operator.add` keeps every
branch's value.

In [ ]:
# ============================================================================
# TWO PARALLEL WRITERS, WITH AND WITHOUT A REDUCER
# ============================================================================
class PlainState(TypedDict):
    request: str
    last_writer: str                              # no reducer: one value per step


class ReducedState(TypedDict):
    request: str
    writers: Annotated[list[str], operator.add]   # reducer: values are concatenated


def writer(name: str, key: str, as_list: bool):
    def node(state) -> dict:
        return {key: [name] if as_list else name}
    return node


def fan_out(schema, key: str, as_list: bool):
    builder = StateGraph(schema)
    for name in ("hotel1_agent", "hotel2_agent"):
        builder.add_node(name, writer(name, key, as_list))
        builder.add_edge(START, name)
        builder.add_edge(name, END)
    return builder.compile()


try:
    fan_out(PlainState, "last_writer", as_list=False).invoke({"request": "Any Deluxe rooms?"})
    print("❌ no error: expected InvalidUpdateError")
except InvalidUpdateError as error:
    print(f"✅ plain key -> InvalidUpdateError: {str(error).splitlines()[0]}")

result = fan_out(ReducedState, "writers", as_list=True).invoke({"request": "Any Deluxe rooms?", "writers": []})
print(f"{'✅' if len(result['writers']) == 2 else '❌'} reducer key -> both values kept: {result['writers']}")

---
## 🔀 Part 3: Attempt 1 - Both Tool Loops Share `messages`

`messages` already has a reducer (`add_messages`), so the first design simply lets both reservation assistants
loop over it: `agent → tools → agent`. The stand-ins below behave like the real thing without an LLM:
- each **fake agent** makes one `list_rooms` tool call, then answers from the tool results it can match to its
  own call ids
- each hotel has a real `ToolNode` with a **same-named** `list_rooms` tool that reports that hotel's rooms

A `ToolNode` answers the tool calls of the **latest** `AIMessage` in its message list. With one shared list,
what is "latest" depends on the other branch too. The check at the end matches every tool result to the call it
answers.

In [ ]:
# ============================================================================
# ATTEMPT 1 - ONE SHARED MESSAGES KEY
# ============================================================================
def make_list_rooms_tool(hotel: str):
    numbers = [room[0] for room in HOTEL_ROOMS[hotel]]

    @tool("list_rooms")
    def list_rooms() -> str:
        """List this hotel's room numbers."""
        return f"{hotel} rooms {numbers}"

    return list_rooms


stub_tools = {hotel: make_list_rooms_tool(hotel) for hotel in HOTELS}


def fake_agent(hotel: str, key: str, rounds: int = 1):
    """Stands in for an LLM tool loop: `rounds` tool calls, then a final answer. No model call."""
    def node(state) -> dict:
        messages = state[key]
        my_calls = [tc["id"] for m in messages if isinstance(m, AIMessage) and m.name == hotel for tc in m.tool_calls]
        if len(my_calls) < rounds:
            call = {"name": "list_rooms", "args": {}, "id": f"{hotel}-call-{len(my_calls) + 1}", "type": "tool_call"}
            return {key: [AIMessage(content="", tool_calls=[call], name=hotel)]}
        results = [m.content for m in messages if isinstance(m, ToolMessage) and m.tool_call_id in my_calls]
        answered = {m.tool_call_id for m in messages if isinstance(m, ToolMessage) and m.tool_call_id in my_calls}
        note = f" ({len(my_calls) - len(answered)} of my calls never got a result)" if len(answered) < len(my_calls) else ""
        return {key: [AIMessage(content=f"{hotel} found: {results}{note}", name=hotel)]}
    return node


class SharedState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    merge_log: Annotated[list[str], operator.add]


def shared_merge(state: SharedState) -> dict:
    return {"merge_log": [f"run {len(state.get('merge_log', [])) + 1}"]}


builder = StateGraph(SharedState)
builder.add_node("reservation_mapper", lambda state: {})
builder.add_node("merge_answers", shared_merge)
builder.add_edge(START, "reservation_mapper")
for hotel in HOTELS:
    builder.add_node(f"{hotel}_agent", fake_agent(hotel, "messages"))
    builder.add_node(f"{hotel}_tools", ToolNode([stub_tools[hotel]]))
    builder.add_edge("reservation_mapper", f"{hotel}_agent")
    builder.add_conditional_edges(f"{hotel}_agent", tools_condition,
                                  {"tools": f"{hotel}_tools", END: "merge_answers"})
    builder.add_edge(f"{hotel}_tools", f"{hotel}_agent")
builder.add_edge("merge_answers", END)

shared = builder.compile().invoke({"messages": [HumanMessage(content="Any Deluxe rooms?")]})

print("Tool results in the shared list (which call they answer <- which hotel's tool produced them):")
crossed = 0
for m in shared["messages"]:
    if isinstance(m, ToolMessage):
        caller, producer = m.tool_call_id.split("-")[0], m.content.split()[0]
        crossed += caller != producer
        print(f"  {'✅' if caller == producer else '❌'} {m.tool_call_id:<15} <- {m.content}")
print("\nFinal answers:")
for m in shared["messages"]:
    if isinstance(m, AIMessage) and not m.tool_calls:
        print(f"  {m.content}")
print(f"\n{'❌' if crossed else '✅'} {crossed} tool result(s) came from the wrong hotel's database")

**Reading the output:** after the first step the shared list ends with **one** hotel's tool call, so **both**
`ToolNode`s answer that same call: one of them with the wrong hotel's rooms, while the other hotel's call gets no
result at all. With the real SQL tools the names match too (`run_query` in both), so the wrong database
answers silently. A real LLM would also see the other branch's calls in its history, and the OpenAI API rejects a
tool call that has no tool result. Both branches need **their own** message list.

---
## 🧵 Part 4: Attempt 2 - One Message Key per Branch

The state gets `messages_hotel1` and `messages_hotel2`:
- the **mapper** copies the conversation into both keys (`RemoveMessage(id=REMOVE_ALL_MESSAGES)` first, so each
  turn starts from a clean copy)
- each agent reads and writes only its own key
- `ToolNode(tools, messages_key="messages_hotel1")` answers calls from that key only
- `partial(tools_condition, messages_key="messages_hotel1")` routes on that key only

`build_branch_stub()` builds this graph with the same fake agents. **Hotel 1 now needs two tool rounds** and
Hotel 2 one, as real agents rarely take the same number of steps. In this attempt each branch's "finished" edge
goes straight to the merge node.

In [ ]:
# ============================================================================
# ATTEMPT 2 - SEPARATE KEYS, MERGE NODE WIRED TO EACH BRANCH
# ============================================================================
class BranchState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    messages_hotel1: Annotated[list[AnyMessage], add_messages]
    messages_hotel2: Annotated[list[AnyMessage], add_messages]
    merge_log: Annotated[list[str], operator.add]


def is_finished(messages) -> bool:
    return bool(messages) and isinstance(messages[-1], AIMessage) and not messages[-1].tool_calls


def branch_merge(state: BranchState) -> dict:
    finished = {hotel: is_finished(state[f"messages_{hotel}"]) for hotel in HOTELS}
    return {"merge_log": [f"run {len(state.get('merge_log', [])) + 1}: finished branches {finished}"]}


def reservation_mapper(state) -> dict:
    """Fan-out: a fresh copy of the conversation in each branch's key (the real system reuses this node)."""
    return {f"messages_{hotel}": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *state["messages"]] for hotel in HOTELS}


def branch_done(state) -> dict:
    """Sync node: runs once, when its branch's tool loop has finished (used in Part 5). Writes nothing."""
    return {}


def build_branch_stub(use_sync_nodes: bool):
    builder = StateGraph(BranchState)
    builder.add_node("reservation_mapper", reservation_mapper)
    builder.add_node("merge_answers", branch_merge)
    builder.add_edge(START, "reservation_mapper")
    for hotel, rounds in (("hotel1", 2), ("hotel2", 1)):
        key = f"messages_{hotel}"
        after_loop = f"{hotel}_done" if use_sync_nodes else "merge_answers"
        builder.add_node(f"{hotel}_agent", fake_agent(hotel, key, rounds))
        builder.add_node(f"{hotel}_tools", ToolNode([stub_tools[hotel]], messages_key=key))
        builder.add_edge("reservation_mapper", f"{hotel}_agent")
        builder.add_conditional_edges(f"{hotel}_agent", partial(tools_condition, messages_key=key),
                                      {"tools": f"{hotel}_tools", END: after_loop})
        builder.add_edge(f"{hotel}_tools", f"{hotel}_agent")
        if use_sync_nodes:
            builder.add_node(f"{hotel}_done", branch_done)
    if use_sync_nodes:
        builder.add_edge([f"{hotel}_done" for hotel in HOTELS], "merge_answers")  # waits for BOTH
    builder.add_edge("merge_answers", END)
    return builder.compile()


def show_branch_run(result) -> None:
    for hotel in HOTELS:
        print(f"{hotel}: {result[f'messages_{hotel}'][-1].content}")
    print("\nMerge-node log:")
    for line in result["merge_log"]:
        print(f"  {line}")
    runs = len(result["merge_log"])
    print(f"\n{'✅' if runs == 1 else '❌'} the merge node ran {runs} time(s); it should run once, after both branches")


no_join = build_branch_stub(use_sync_nodes=False).invoke({"messages": [HumanMessage(content="Any Deluxe rooms?")]})
show_branch_run(no_join)

**Reading the output:** each hotel's answer now contains only its own rooms, so the keys are separated. But the
merge node ran **twice**: first as soon as Hotel 2 finished (its log line shows Hotel 1 still unfinished), then again
for Hotel 1. A conditional edge fires the merge node every time **any** branch gets there. In the real system that
means two replies to the guest, the first one missing a hotel.

---
## 🤝 Part 5: Attempt 3 - Sync Nodes and a Join Edge

`builder.add_edge(["hotel1_done", "hotel2_done"], "merge_answers")` is a **join**: the merge node runs only
after *both* listed nodes have run. The join can't be drawn from the agent nodes themselves, because they also run
in the middle of their tool loops; so each branch ends in a small `*_done` **sync node** that runs exactly once,
when its loop has finished.

In [ ]:
# ============================================================================
# ATTEMPT 3 - SYNC NODES + JOIN EDGE
# ============================================================================
join_graph = build_branch_stub(use_sync_nodes=True)
print("Edges:")
hc.print_edges(join_graph)
print()
show_branch_run(join_graph.invoke({"messages": [HumanMessage(content="Any Deluxe rooms?")]}))

**Reading the output:** the edges list shows `hotel1_done` and `hotel2_done` both leading into
`merge_answers`, and the merge node ran **once**, with both branches finished. This is the shape of the real
system.

---
## 🏗️ Part 6: The Real System

The core comes from `hotel_common`: 03's coordinator guideline and compliance checker (`retrieve` + `generate`,
with 04's structured verdict). Where the coordinator used to forward to one `reservation_assistant`, it now
reaches the **mapper**:

`conv_assistant → reservation_mapper → hotel{1,2}_assistant ⇄ hotel{1,2}_tools → hotel{1,2}_done → merge_answers`

- `make_hotel_assistant(hotel)` builds both assistants from **one** factory, so each is bound to its own hotel's
  tools. (Copy-pasting the first assistant to make the second is how Hotel 2's assistant ends up calling
  Hotel 1's tools, and silently answering from the wrong database.)
- Both run the same query at the same time, so if both booked, the guest would get **two** reservations. Their
  tools are read-only (no `book_room`), so they can only report availability and prices; booking comes once the
  guest has picked a hotel.
- The merge node simply labels and joins the two answers.
  `07_hotel_merging_and_conversation_memory.ipynb` lets an LLM merge them.

The parallel pieces (`reservation_mapper`, `branch_done`, `make_hotel_assistant`) are written out in this notebook;
`hotel_common` keeps the same versions for `07_hotel_merging_and_conversation_memory.ipynb`.

In [ ]:
# ============================================================================
# STATE, AGENT NODES AND ROUTER
# ============================================================================
class ParallelState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]          # the guest conversation
    messages_hotel1: Annotated[list[AnyMessage], add_messages]   # Hotel 1 assistant's private tool loop
    messages_hotel2: Annotated[list[AnyMessage], add_messages]   # Hotel 2 assistant's private tool loop


AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(
    AGENTS,
    extra_guidelines=hc.COMPLIANCE_FIRST + "\n"  # 03's guideline, plus:
    "- The chain has two hotels. reservation_assistant checks both of them at once.",
)
# 03's compliance checker with 04's verdict; no single reservation agent: the hotel branches replace it
_, rag_assistant, generate = hc.make_core_nodes(llm, [], rag_tools)


def conv_assistant(state: ParallelState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + state["messages"])]}


# --- the parallel reservation branch ---
# reservation_mapper and branch_done: defined in Part 4


def make_hotel_assistant(hotel: str):
    """One factory for both assistants, so each is bound to its OWN hotel's tools and key."""
    key = f"messages_{hotel}"
    numbers = ", ".join(str(room[0]) for room in HOTEL_ROOMS[hotel])
    prompt = SystemMessage(content=hc.SQL_READ_PROMPT.content + f"""

You are one of several reservation assistants answering in parallel, one per hotel of the chain. You handle
{HOTELS[hotel]} only (rooms {numbers}). Report availability and prices for your hotel. You cannot book: the guest
first chooses a hotel.""")
    llm_with_tools = llm.bind_tools(sql_tools[hotel])

    def hotel_assistant(state: ParallelState) -> dict:
        return {key: [llm_with_tools.invoke([prompt] + state[key])]}

    return hotel_assistant


def choose_next_node(state: ParallelState) -> Literal["reservation_mapper", "compliance_checker", "__end__"]:
    target = hc.route_by_mention(state["messages"], AGENTS)  # 03's router, with its loop guard
    return "reservation_mapper" if target == "reservation_assistant" else (target or END)


print("✅ Nodes defined:", ", ".join(["conv_assistant", "rag_assistant", "generate", "reservation_mapper"]
                                    + [f"{hotel}_assistant" for hotel in HOTELS]))

In [ ]:
# ============================================================================
# MERGE NODE AND GRAPH
# ============================================================================
def merge_answers(state: ParallelState) -> dict:
    """Fan-in: label and join the final answer of each branch."""
    parts = [f"{HOTELS[hotel]}: {state[f'messages_{hotel}'][-1].content}" for hotel in HOTELS]
    return {"messages": [AIMessage(content="\n\n".join(parts), name="merge_answers")]}


builder = StateGraph(ParallelState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("compliance_checker", rag_assistant)
builder.add_node("generate", generate)
builder.add_node("rag_tools", ToolNode(rag_tools))
builder.add_node("reservation_mapper", reservation_mapper)
builder.add_node("merge_answers", merge_answers)

builder.add_edge(START, "conv_assistant")
builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_mapper", "compliance_checker", END])
builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
builder.add_edge("rag_tools", "generate")
builder.add_edge("generate", "conv_assistant")

for hotel in HOTELS:
    key = f"messages_{hotel}"
    builder.add_node(f"{hotel}_assistant", make_hotel_assistant(hotel))
    builder.add_node(f"{hotel}_tools", ToolNode(sql_tools[hotel], messages_key=key))
    builder.add_node(f"{hotel}_done", branch_done)
    builder.add_edge("reservation_mapper", f"{hotel}_assistant")
    builder.add_conditional_edges(f"{hotel}_assistant", partial(tools_condition, messages_key=key),
                                  {"tools": f"{hotel}_tools", END: f"{hotel}_done"})
    builder.add_edge(f"{hotel}_tools", f"{hotel}_assistant")
builder.add_edge([f"{hotel}_done" for hotel in HOTELS], "merge_answers")
builder.add_edge("merge_answers", END)

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## ▶️ Part 7: One Question, Two Hotels

The guest asks whether a Deluxe room is free in two months. Both hotels have one (104 and 108), so both
assistants should find something. `hc.run_traced()` streams the run and records every node that ran, so the
checks below are computed from the run itself: which branches ran, which SQL each one executed against its own
database, how often the merge node ran, and whether anything was written.

In [ ]:
# ============================================================================
# SCRIPTED QUESTION AND CODE-CHECKED FAN-OUT/FAN-IN
# ============================================================================
stay_in = date.today() + timedelta(days=60)
stay_out = stay_in + timedelta(days=2)
bookings_before = {hotel: hc.reservation_count(db_paths[hotel]) for hotel in HOTELS}

nodes_run, state = hc.run_traced(graph, f"Hi! Is a Deluxe room available from {stay_in} to {stay_out}? "
                                        "What would it cost per night?", thread_id="parallel-demo")
print("Nodes, in the order they ran:\n  " + " → ".join(nodes_run) + "\n")
print(f"🤵 Reply to the guest:\n{state['messages'][-1].content}\n")

print("--- What each branch did ---")
for hotel in HOTELS:
    branch = state.get(f"messages_{hotel}", [])
    queries = [tc["args"].get("sql", "") for m in branch if isinstance(m, AIMessage)
               for tc in m.tool_calls if tc["name"] == "run_query"]
    truth = hc.free_rooms(db_paths[hotel], "Deluxe", stay_in, stay_out)
    print(f"{HOTELS[hotel]} (ground truth: free Deluxe rooms {truth})")
    for query in queries:
        print(f"   SQL: {' '.join(query.split())}")
    print(f"   {'✅' if f'{hotel}_assistant' in nodes_run else '❌'} assistant ran "
          f"{nodes_run.count(f'{hotel}_assistant')} time(s); "
          f"{'✅' if queries else '❌'} ran {len(queries)} query(ies) on {db_paths[hotel]}")

print("\n--- Fan-in and side effects ---")
merge_runs = nodes_run.count("merge_answers")
print(f"{'✅' if merge_runs == 1 else '❌'} merge node ran {merge_runs} time(s) (expected exactly 1)")
for hotel in HOTELS:
    after = hc.reservation_count(db_paths[hotel])
    print(f"{'✅' if after == bookings_before[hotel] else '❌'} {db_paths[hotel]} reservations unchanged ({after})")

**Reading the output:** the node trace shows the coordinator, the compliance check, then both
`hotel*_assistant` nodes interleaved with their tool nodes, and a single `merge_answers` at the end. Under
each hotel you see the SQL it ran against its own database; compare the reply's room numbers with the ground
truth printed next to it (104 for Hotel 1, 108 for Hotel 2). If the coordinator did not forward (❌ on both
assistants), read its reply: it is asking for something it considers missing.

To chat with the system yourself, run `hc.chat_with_agent(graph, "my-thread")` in a new cell.

---
## 📝 Summary

- A key written by **parallel nodes** needs a **reducer**; a plain key raises `InvalidUpdateError`.
- Parallel tool loops must not share `messages`: a `ToolNode` answers the **latest** tool call in its list,
  which may belong to the other branch. Give each branch its own key, with `ToolNode(..., messages_key=...)` and
  `partial(tools_condition, messages_key=...)`.
- Wiring each branch straight to the merge node runs it once **per branch**. Sync nodes plus
  `add_edge([a_done, b_done], "merge_answers")` make it wait for both.
- Build parallel agents from **one factory** so each gets its own tools, and give them read-only tools when a write
  must happen only once.

### Next Steps
- `07_hotel_merging_and_conversation_memory.ipynb`: an LLM merges the two answers by priority, and a long
  conversation is trimmed and summarized